# Entrenar y evaluar PredictToDo con datos locales

Este ejemplo muestra el ciclo básico: cargar datos preprocesados, separar periodos,
crear un modelo, optimizarlo y evaluar sus predicciones. Usa directamente
`TrafficDataset` y `GNN_Temporal_Model` de `src`. Se puede ejecutar en un equipo
local o en cualquier entorno Python compatible, sin servicios externos.

**Datos incluidos:** 5.000 vías conectadas desde una semilla en la calle 12A,
con intervalos de 15 minutos del 1 al 15 de octubre de 2024 (45,75 MB).
La [ficha de la muestra](../examples/bogota/README.md) describe su alcance.
Se entrena desde cero sobre 32 vías del paquete por defecto.
El entrenamiento corto enseña el procedimiento; no reproduce resultados de un
modelo desplegado ni permite concluir que el modelo generaliza a toda la ciudad.

Se necesita Python 3.12 y las dependencias del proyecto instaladas en el kernel.

## 1. Configurar los datos

Se asume el formato que consume `TrafficDataset`:

| Elemento | Estructura esperada |
|---|---|
| Grafo | Grafo NetworkX serializado; identificadores de vías y orden de nodos coherentes con los metadatos |
| Atributos estáticos | HDF5, matriz `data`: filas `(índice de nodo, índice de variable, valor)` |
| Variables dinámicas y objetivo | HDF5, matriz `data`: filas `(índice de nodo, índice temporal, índice de variable, valor)`, ordenadas por tiempo |
| Metadatos | Diccionarios `node_idx_map`, `time_idx_map` y `feature_idx_map`; cada HDF5 tiene un archivo acompañante con sufijo `_metadata.pkl` |
| Objetivo | Variable `mean_jam_level` |

Los índices empiezan en cero. Los archivos comparten los mismos nodos y tiempos.
El ejemplo utiliza intervalos de 15 minutos y un solo subgrafo fijo. Las variables
futuras deben ser conocidas al emitir el pronóstico: no incluir observaciones
futuras de tráfico. El calendario se añade automáticamente por el cargador.

Las entradas ausentes se rellenan a cero por el cargador como simplificación
computacional; esos ceros no acreditan una observación. Para el objetivo, una
máscara identifica las filas explícitas de `mean_jam_level`: solo estas participan
en pérdida, métricas y referencia constante, incluidos los ceros explícitos.
La evaluación queda condicionada a la presencia de registros, no mide toda la red. Los archivos pickle deben proceder de una fuente de confianza.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src/sparse_dataloader.py").is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abre el notebook desde la carpeta del proyecto o indica PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "examples" / "bogota" / "data"
GRAPH = DATA_DIR / "graph.pkl"
STATIC = DATA_DIR / "static_dataset.h5"
DYNAMIC = [DATA_DIR / f"{name}_dataset.h5" for name in [
    "coi_points", "coi_tracts", "events", "incidents", "irregularities",
    "jam_delay", "jam_level_count", "jam_velocity",
]]
TARGET = DATA_DIR / "jams_dataset.h5"
FUTURE_FEATURES = []  # Solo variables conocidas de antemano; calendario automático.
INPUT_STEPS = 12
FORECAST_STEPS = 4
MAX_NODES = 32
EPOCHS = 2
MAX_WINDOWS = 16  # Por partición: limita el coste de esta demostración.
OUTPUT_DIR = PROJECT_ROOT / "data" / "example_output"

In [ ]:
import hashlib
import h5py
import copy
import json
import pickle
import numpy as np
import networkx as nx
from itertools import islice
import pandas as pd
import torch
from src.sparse_dataloader import TrafficDataset
from src.models.gnn_temporal import GNN_Temporal_Model

torch.manual_seed(7)
np.random.seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def metadata_path(path):
    return path.with_name(path.stem + "_metadata.pkl")

required = [GRAPH, STATIC, TARGET, *DYNAMIC,
            *[metadata_path(p) for p in [STATIC, TARGET, *DYNAMIC]]]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Incorpora los datos preprocesados y ajusta la configuración. Faltan:\n" + "\n".join(missing))

manifest = json.loads((DATA_DIR.parent / "manifest.json").read_text())
for name, record in manifest["files"].items():
    assert hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() == record["sha256"], name

with GRAPH.open("rb") as f:
    graph = pickle.load(f)
metadata = {}
for path in [STATIC, *DYNAMIC, TARGET]:
    with metadata_path(path).open("rb") as f:
        metadata[str(path)] = pickle.load(f)
node_map = {node: i for i, node in enumerate(graph.nodes)}
assert node_map, "El grafo debe contener vías."
for meta in metadata.values():
    assert meta["node_idx_map"] == node_map, "Los nodos y su orden deben coincidir con el grafo."
target_meta = metadata[str(TARGET)]
for path in DYNAMIC:
    assert metadata[str(path)]["time_idx_map"] == target_meta["time_idx_map"], "Alinea los tiempos de todos los conjuntos."
times = pd.DatetimeIndex([t for t, i in sorted(target_meta["time_idx_map"].items(), key=lambda x: x[1])])
assert list(sorted(target_meta["time_idx_map"].values())) == list(range(len(times)))
assert times.is_monotonic_increasing and times.is_unique
assert (times[1:] - times[:-1] == pd.Timedelta(minutes=15)).all(), "Se esperan intervalos regulares de 15 min."
assert "mean_jam_level" in target_meta["feature_idx_map"]
print("Datos encontrados. Dispositivo:", device)

## 2. Construir ventanas y separar periodos

`TrafficDataset` transforma los archivos en tensores estáticos, históricos y
futuros sobre un subgrafo. Aquí se desactivan las entradas opcionales de clima e
imputación para mantener el contrato mínimo; no se intenta reproducir la
configuración de ningún modelo existente.

La división es cronológica, 60 % / 20 % / 20 %. Cada ventana completa —entrada y
objetivo— queda dentro de su periodo, evitando solapamientos entre entrenamiento,
validación y prueba. La muestra debe ser suficientemente larga para los tres.
Se eligen hasta 16 ventanas repartidas por cada partición, con al menos un objetivo
explícito por ventana. La máscara se construye para el subgrafo fijo de 32 vías;
si se cambia la semilla, hay que volver a ejecutar las celdas desde el inicio.


In [ ]:
seed = next((n for n in graph if next(iter(graph.neighbors(n)), None) is not None), None)
if seed is None:
    raise ValueError("El ejemplo necesita al menos dos vías conectadas.")
# Evitar rellenar el subgrafo con copias de la misma vía si la muestra es pequeña.
node_count = 1 + len(list(islice(nx.bfs_edges(graph, seed), MAX_NODES - 1)))
dataset = TrafficDataset(
    graph_path=str(GRAPH), static_data_path=str(STATIC),
    dynamic_data_paths=[str(p) for p in DYNAMIC],
    metadata_paths=[str(metadata_path(p)) for p in DYNAMIC],
    target_data_path=str(TARGET), in_timesteps=INPUT_STEPS,
    out_timesteps=FORECAST_STEPS, future_features=FUTURE_FEATURES,
    node_batch_size=node_count, node_seed=seed,
    include_weather=False, include_velocity_target=False,
    normalization_constants_path=None,
)
T = dataset.effective_total_timesteps
window = INPUT_STEPS + FORECAST_STEPS
cut1, cut2 = int(T * 0.6), int(T * 0.8)

# La máscara usa el mismo orden de vías que TrafficDataset para esta semilla.
selected_nodes = dataset.batch_mapping[seed]
selected_ids = np.array([target_meta["node_idx_map"][node] for node in selected_nodes])
row_for_node = {int(node): row for row, node in enumerate(selected_ids)}
target_present = np.zeros((node_count, T), dtype=bool)
with h5py.File(TARGET, "r") as f:
    rows = f["data"][:]
level_rows = rows[rows[:, 2] == target_meta["feature_idx_map"]["mean_jam_level"]]
for node, t, _, value in level_rows:
    row = row_for_node.get(int(node))
    if row is not None and np.isfinite(value):
        target_present[row, int(t)] = True

def target_mask(i):
    return torch.from_numpy(target_present[:, i + INPUT_STEPS:i + window]).to(device)

def starts(a, b):
    candidates = [i for i in range(a, b - window + 1, window)
                  if target_present[:, i + INPUT_STEPS:i + window].any()]
    if not candidates:
        raise ValueError("No hay ventanas con objetivos explícitos en esta partición.")
    positions = np.linspace(0, len(candidates) - 1, min(MAX_WINDOWS, len(candidates)), dtype=int)
    return [candidates[j] for j in positions]

train_ids, val_ids, test_ids = starts(0, cut1), starts(cut1, cut2), starts(cut2, T)
assert max(train_ids) + window <= min(val_ids)
assert max(val_ids) + window <= min(test_ids)
print({"entrenamiento": len(train_ids), "validación": len(val_ids), "prueba": len(test_ids)})
sample = dataset[train_ids[0]]
print({k: tuple(sample[k].shape) for k in ["x_static", "x_dynamic_past", "x_dynamic_future", "edge_index", "y"]})

## 3. Normalizar y crear el modelo

Para esta demostración calculamos media y desviación por variable **solo con las
ventanas de entrenamiento**. Aplicamos la misma transformación a validación y
prueba. El objetivo conserva su escala original, por lo que el MAE se expresa en
unidades de nivel de congestión. Estos parámetros de ejemplo no son compatibles
por defecto con pesos entrenados con otra normalización.

In [ ]:
feature_keys = ["x_static", "x_dynamic_past", "x_dynamic_future"]
scalers = {}
for key in feature_keys:
    values = torch.cat([dataset[i][key].reshape(-1, sample[key].shape[-1]) for i in train_ids])
    assert torch.isfinite(values).all(), f"Hay valores no finitos en {key}."
    scalers[key] = (values.mean(0).to(device), values.std(0, unbiased=False).clamp_min(1e-6).to(device))

model_config = dict(
    static_dim=sample.x_static.shape[-1],
    dynamic_dim=sample.x_dynamic_past.shape[-1],
    future_dim=sample.x_dynamic_future.shape[-1],
    input_seq_len=INPUT_STEPS, forecast_horizon=FORECAST_STEPS,
    model_type="GConvGRU", include_velocity_target=False,
)
model = GNN_Temporal_Model(edge_index=sample.edge_index, **model_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

def predict_sample(item):
    normalized = {key: (item[key].to(device) - scalers[key][0]) / scalers[key][1]
                  for key in feature_keys}
    prediction = model(
        item.edge_index.to(device), normalized["x_static"],
        normalized["x_dynamic_past"].permute(1, 0, 2).unsqueeze(0),
        normalized["x_dynamic_future"].permute(1, 0, 2).unsqueeze(0),
    )[0, :, :, 0]
    target = item.y.to(device)
    assert prediction.shape == target.shape
    assert torch.isfinite(prediction).all() and torch.isfinite(target).all()
    return prediction, target

## 4. Entrenar y elegir una versión

Cada paso usa una ventana del mismo subgrafo: predecir, calcular el error,
propagar gradientes y actualizar pesos. Validación selecciona la versión con
menor MAE. El periodo de prueba permanece reservado hasta terminar la selección.
Para entrenar sobre toda una red se amplía la cobertura de subgrafos y se conserva
la correspondencia de nodos, aristas y variables de cada muestra.

In [ ]:
@torch.no_grad()
def evaluate(indices):
    model.eval()
    absolute_error = 0.0
    count = 0
    for i in indices:
        prediction, target = predict_sample(dataset[i])
        mask = target_mask(i)
        absolute_error += (prediction[mask] - target[mask]).abs().sum().item()
        count += int(mask.sum().item())
    return absolute_error / count

history = []
best_mae = float("inf")
best_state = None
for epoch in range(EPOCHS):
    model.train()
    losses = []
    for i in train_ids:
        optimizer.zero_grad(set_to_none=True)
        prediction, target = predict_sample(dataset[i])
        mask = target_mask(i)
        loss = loss_fn(prediction[mask], target[mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    val_mae = evaluate(val_ids)
    history.append({"epoch": epoch + 1, "train_mse": float(np.mean(losses)), "val_mae": val_mae})
    if val_mae < best_mae:
        best_mae = val_mae
        best_state = copy.deepcopy(model.state_dict())
assert best_state is not None
model.load_state_dict(best_state)
pd.DataFrame(history)

## 5. Evaluar y observar una predicción

El MAE resume el error absoluto medio sobre los objetivos explícitos de las vías y horizontes de prueba.
Una comparación básica usa el nivel medio observado durante entrenamiento como
predicción constante. Conviene ampliar el análisis por horizonte, nivel de
congestión y zona, y comparar con persistencia cuando la muestra incluya esa señal.
Estas métricas describen únicamente las ventanas y el subgrafo seleccionados.

In [ ]:
test_mae = evaluate(test_ids)
train_target = torch.cat([dataset[i].y[target_mask(i).cpu()] for i in train_ids])
baseline = train_target.mean().item()
error_sum, count = 0.0, 0
for i in test_ids:
    y = dataset[i].y
    mask = target_mask(i).cpu()
    error_sum += (y[mask] - baseline).abs().sum().item()
    count += int(mask.sum().item())
print({"test_mae_explicit_targets": test_mae, "explicit_test_targets": count, "baseline_mean_mae": error_sum / count})
with torch.no_grad():
    prediction, target = predict_sample(dataset[test_ids[0]])
mask = target_mask(test_ids[0]).cpu()
example_node = int(torch.where(mask.any(dim=1))[0][0])
example = pd.DataFrame({
    "pasos_desde_el_origen": range(1, FORECAST_STEPS + 1),
    "observado": np.where(mask[example_node].numpy(), target[example_node].cpu().numpy(), np.nan),
    "predicho": prediction[example_node].cpu().numpy(),
})
example

## 6. Guardar lo necesario para repetir la inferencia

Los pesos se acompañan de la arquitectura, las transformaciones y la descripción
de las variables. En un uso real también se debe conservar la versión de los datos
y el grafo. La siguiente celda comprueba que recargar los pesos mantiene la
predicción de referencia. Los archivos de salida permanecen locales.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint = OUTPUT_DIR / "example_model.pt"
torch.save({
    "state_dict": model.state_dict(), "model_config": model_config,
    "scalers": {k: (v[0].cpu(), v[1].cpu()) for k, v in scalers.items()},
}, checkpoint)
contract = {
    "data_manifest_sha256": hashlib.sha256((DATA_DIR.parent / "manifest.json").read_bytes()).hexdigest(),
    "target_policy": "explicit mean_jam_level rows only; missing inputs zero-filled",
    "example_only": True, "seed": str(seed), "max_nodes": node_count,
    "future_features": FUTURE_FEATURES, "input_steps": INPUT_STEPS,
    "forecast_steps": FORECAST_STEPS, "target": "mean_jam_level",
    "feature_maps": {Path(k).name: v["feature_idx_map"] for k, v in metadata.items()},
    "temporal_features": dataset.temporal_feature_names,
    "window_starts": {"train": train_ids, "validation": val_ids, "test": test_ids},
}
(OUTPUT_DIR / "example_contract.json").write_text(json.dumps(contract, indent=2, ensure_ascii=False))
loaded = torch.load(checkpoint, map_location=device, weights_only=True)
restored = GNN_Temporal_Model(edge_index=sample.edge_index, **loaded["model_config"]).to(device)
restored.load_state_dict(loaded["state_dict"])
restored.eval()
original = model
original_scalers = scalers
try:
    model = restored
    scalers = {k: (v[0].to(device), v[1].to(device)) for k, v in loaded["scalers"].items()}
    with torch.no_grad():
        reloaded_prediction, _ = predict_sample(dataset[test_ids[0]])
    torch.testing.assert_close(reloaded_prediction, prediction)
finally:
    model = original
    scalers = original_scalers
print("Modelo guardado y recarga comprobada.")

## Siguientes pasos

La muestra permite comprobar el ciclo completo con datos reales. Para un análisis
predictivo, revisar la cobertura y el tratamiento de entradas ausentes, ampliar
ventanas y subgrafos, elegir parámetros con validación y reservar la prueba para
la evaluación final. Las métricas de este ejemplo no representan toda Bogotá.

Para profundizar: [cargador de datos](../src/sparse_dataloader.py),
[modelos](../src/models/gnn_temporal.py), [funciones de pérdida](../src/training/losses.py)
y [pruebas del cargador](../tests/test_sparse_dataloader.py).